# VEO Tracking — Pipeline completo alpha=1+crop

Pipeline de tracking con preprocesado definitivo alpha=1+crop: undistort + crop antes de YOLO, homografias calibradas sobre frames corregidos.

## Requisitos externos
- **Video panoramico VEO** (`data/videos/veo_panoramico_banyoles.mp4`) — descargar con yt-dlp
- **Modelo YOLO jugadores** (`MODEL_PATH` en CONFIG) — entrenar con `train_yolo.py`
- **Homografias**: si no existen, carga automaticamente `data/example_banyoles/H_camA.npy` y `H_camB.npy`


# 🎯 VEO Tracking — alpha=1+crop

**TFG Análisis Táctico — Cámara Panorámica VEO**

Pipeline completo con preprocesado definitivo **alpha=1+crop**:
- Cada frame se pasa por elastic undo + undistort (alpha=1) + crop antes de YOLO
- Las homografías son las calibradas directamente sobre frames alpha=1+crop
- Tracker: ByteTrack (×2 cámaras) → seam dedup → KalmanFieldTracker
- Clasificación por color BGR + PKL día/noche


## 1 · Importaciones

In [13]:
import cv2
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle, Circle, Arc
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import display as _disp
import ipywidgets as widgets
import os
from ultralytics import YOLO

import sys as _sys
_sys.path.insert(0, '.')
from veo_pipeline import undo_elastic_compression, K_LEFT, K_RIGHT, DIST_COEFFS
from pipeline_core import classify_gk_score


## 2 · ⚙️ Configuración
Edita **solo este bloque** para adaptar el notebook a tu entorno.
Cambia `VIDEO_PATH`, las rutas de homografía, el modelo YOLO y los PKLs.

In [14]:
# ═══════════════════════════════════════════════════════════
#  CONFIGURACION — edita solo este bloque
# ═══════════════════════════════════════════════════════════

# --- Rutas -------------------------------------------------------
VIDEO_PATH       = "data/videos/veo_panoramico_banyoles.mp4"
H_A_PATH         = "data/calib_alpha1/calib_camA_alpha1_homography.npy"
H_B_PATH         = "data/calib_alpha1/calib_camB_alpha1_homography.npy"
# --- Preprocesado alpha=1+crop -------------------------------------------
# Las homografías ya están calibradas sobre frames alpha=1+crop,
# por lo que se usan directamente (sin ajuste de T).
# Los mapas de undistort se pre-computan en la celda de preprocesado.
ALPHA_UNDIST = 1.0
MODEL_PLAYERS    = "runs/detect/modelo_players_v24_panoramic2/weights/best.pt"

PROTO_DAY_PATH   = "prototypes_v3_day.pkl"
PROTO_NIGHT_PATH = "prototypes_v3_night.pkl"
PROTO_PATH       = "prototypes_v3.pkl"          # fallback si no hay dual

# --- Campo -------------------------------------------------------
L_M   = 98.0    # longitud (m)
A_M   = 61.0    # anchura  (m)
ESCALA = 10     # px por metro en el mapa 2D

# --- Modelos -----------------------------------------------------
MODEL_BALL       = "runs/detect/modelo_ball_v33/weights/best.pt"

# --- Deteccion ---------------------------------------------------
# Sube CONF_THRESH si hay demasiadas detecciones falsas (espectadores,
# banquillo, fotografos). Con 0.50-0.55 se eliminan la mayoria.
CONF_THRESH        = 0.52   # subido de 0.45 para filtrar falsos positivos
CONF_BALL          = 0.40
BRIGHT_THRESH      = 75.0   # umbral brillo dia/noche
CLASSIFY_MAX_SCORE = 80.0   # score color > umbral → clase 'unknown'

# --- Filtros post-deteccion --------------------------------------
# NMS global: si 2 detecciones estan a menos de GLOBAL_NMS_M metros
# en el campo se fusionan (la de mayor confianza se queda).
# Elimina duplicados de la costura que escapan al seam dedup.
GLOBAL_NMS_M   = 2.5    # metros minimos entre detecciones distintas
MAX_DETS_FRAME = 26     # maximo de jugadores por frame (11+11+ref+3 margen)

# --- Costura A/B (seam deduplication) ----------------------------
SEAM_MARGIN_M  = 6.0   # margen ampliado (era 4.0) — mas solapamiento cubierto
SEAM_FUSION_M  = 2.5   # distancia ampliada (era 1.5) — absorbe error de homografia

# --- Tracking ----------------------------------------------------
TRACK_MAX_DIST_M  = 6.0
TRACK_WINDOW_S    = 7.0   # segundos sin deteccion antes de expirar un ID
TRACK_CLASS_W     = 8.0   # penalizacion (m) si clase no coincide

# --- Porteros y arbitros --------------------------------
HOME_ATTACKS_LEFT = True   # True si local ataca hacia porteria izquierda

# --- Segmento a procesar -----------------------------------------
START_FRAME = 18000   # frame de inicio  (cambia libremente)
N_FRAMES    = 900     # frames a procesar  (900 = 30 s a 30 fps)

# --- Colores -----------------------------------------------------
CLASSES_ORDER = ['player_home', 'player_away', 'gk_home', 'gk_away', 'referee']
COLOR_MAP_BGR = {
    'player_home': (0,   0,   220),
    'player_away': (220, 60,  0),
    'gk_home':     (230, 0,   230),
    'gk_away':     (0,   200, 200),
    'referee':     (0,   220, 220),
}
COLOR_MAP_HEX = {
    'player_home': '#DC143C',
    'player_away': '#1E90FF',
    'gk_home':     '#FF00FF',
    'gk_away':     '#00CED1',
    'referee':     '#FFD700',
}
TEAM_COLORS = COLOR_MAP_HEX
BALL_COLOR  = '#FF6B35'
DARK_BG     = '#0d1117'
GRASS_DARK  = '#1a472a'
GRASS_LIGHT = '#215732'
LINE_COLOR  = 'white'
# -- Fallbacks a datos de ejemplo ----------------------------------
import os as _os
if not _os.path.exists(H_A_PATH): H_A_PATH = "data/example_banyoles/H_camA.npy"
if not _os.path.exists(H_B_PATH): H_B_PATH = "data/example_banyoles/H_camB.npy"


## 3 · Funciones auxiliares
Descriptor de color BGR, clasificación, iluminación y dibujo del campo.

In [15]:
# ═══════════════════════════════════════════════════════════
#  FUNCIONES AUXILIARES
#  Descriptor BGR · Clasificacion · Iluminacion · Campo 2D
# ═══════════════════════════════════════════════════════════

# --- Descriptor de color BGR (identico a generar_prototipos_dual.py) ---------

def get_torso_crop(crop):
    h, w = crop.shape[:2]
    return crop[int(h * 0.20):int(h * 0.60), int(w * 0.20):int(w * 0.80)]


def get_player_color(crop, grass_hue=None):
    crop_rs = cv2.resize(crop, (64, 64))
    hsv = cv2.cvtColor(crop_rs, cv2.COLOR_BGR2HSV)
    H = hsv[:, :, 0].astype(np.float32)
    S = hsv[:, :, 1].astype(np.float32)
    V = hsv[:, :, 2].astype(np.float32)
    mask_cesped = (H > 35) & (H < 85) & (S > 50)
    mask_piel   = (H < 20) & (S > 30) & (V > 90)
    mask        = ~(mask_cesped | mask_piel)
    pixels = crop_rs.reshape(-1, 3).astype(np.float32)
    valid  = pixels[mask.flatten()]
    gray   = np.array([128., 128., 128.])
    if len(valid) < 9:
        return gray, gray, [gray.copy(), gray.copy(), gray.copy()]
    median_c = np.median(valid, axis=0)
    mean_c   = np.mean(valid,   axis=0)
    k        = min(3, len(valid))
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(valid, k, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS)
    counts = np.bincount(labels.flatten(), minlength=k)
    order  = np.argsort(-counts)
    top3   = [centers[i] for i in order]
    while len(top3) < 3:
        top3.append(median_c.copy())
    return median_c, mean_c, top3


def color_distance(c1, c2):
    return float(np.linalg.norm(np.array(c1) - np.array(c2)))


def classify_player(crop, class_prototypes, grass_hue=None,
                    max_score=None):
    """
    Clasifica un jugador por color BGR.

    Si best_score > max_score (CLASSIFY_MAX_SCORE por defecto),
    devuelve ('unknown', score) para no asignar una clase dudosa
    (sombras, lineas de banda, detecciones parciales...).
    """
    _threshold = max_score if max_score is not None else CLASSIFY_MAX_SCORE
    median_c, mean_c, top3 = get_player_color(crop, grass_hue)
    best_class, best_score = None, float('inf')
    for cls, proto in class_prototypes.items():
        d_median = color_distance(median_c, proto['median'])
        d_mean   = color_distance(mean_c,   proto['mean'])
        d_cluster = (
            sum(min(color_distance(c, pc) for pc in proto['top3']) for c in top3) / len(top3)
            if proto.get('top3') else 999
        )
        score = 0.25 * d_median + 0.5 * d_mean + 0.25 * d_cluster
        if score < best_score:
            best_score = score
            best_class = cls
    if best_score > _threshold:
        return 'unknown', best_score
    return best_class, best_score


# --- Cesped y luz -------------------------------------------------------------

def get_grass_hue(frame):
    hsv  = cv2.cvtColor(cv2.resize(frame, (320, 180)), cv2.COLOR_BGR2HSV)
    mask = ((hsv[:, :, 0] > 35) & (hsv[:, :, 0] < 85)
            & (hsv[:, :, 1] > 40) & (hsv[:, :, 2] > 40))
    if mask.sum() < 100:
        return None
    return float(np.median(hsv[:, :, 0][mask]))


def frame_brightness(frame):
    return float(np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)))


def detect_lighting(frame, bright_thresh=None):
    thresh = bright_thresh if bright_thresh is not None else BRIGHT_THRESH
    return 'day' if frame_brightness(frame) > thresh else 'night'


def select_protos(frame, protos_day, protos_night, bright_thresh=None):
    if protos_day is None and protos_night is None:
        raise ValueError('Ningun PKL cargado')
    if protos_day   is None: return protos_night
    if protos_night is None: return protos_day
    cond = detect_lighting(frame, bright_thresh)
    return protos_day if cond == 'day' else protos_night


# --- Homografia --------------------------------------------------------------

def esquinas_metros(H, w_px, h_px):
    pts = np.array([[[0, 0], [w_px, 0], [w_px, h_px], [0, h_px]]], dtype=np.float32)
    return cv2.perspectiveTransform(pts, H)[0]


def pixel_to_field(x_px, y_px, H):
    pt = np.array([[[float(x_px), float(y_px)]]], dtype=np.float32)
    pm = cv2.perspectiveTransform(pt, H)[0][0]
    return float(pm[0]), float(pm[1])


# --- Dibujo del campo (matplotlib) ------------------------------------------

def draw_pitch_mpl(ax, lm=None, am=None, limite_x=None):
    lm = lm if lm is not None else L_M
    am = am if am is not None else A_M
    ax.set_facecolor(GRASS_DARK)
    ax.set_xlim(-1, lm + 1)
    ax.set_ylim(am + 1, -1)   # y=0 arriba (coord imagen)
    ax.set_aspect('equal')
    ax.axis('off')

    stripe_w = lm / 12
    for i in range(12):
        c = GRASS_LIGHT if i % 2 == 0 else GRASS_DARK
        ax.add_patch(Rectangle((i * stripe_w, 0), stripe_w, am, color=c, zorder=0))

    lw = 1.4
    kw = dict(color=LINE_COLOR, lw=lw, zorder=2)

    def line(x0, y0, x1, y1): ax.plot([x0, x1], [y0, y1], **kw)
    def rect(x, y, w, h):     ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=LINE_COLOR, lw=lw, zorder=2))
    def circle(cx, cy, r):    ax.add_patch(Circle((cx, cy), r, fill=False, edgecolor=LINE_COLOR, lw=lw, zorder=2))
    def dot(cx, cy, r=0.4):   ax.add_patch(Circle((cx, cy), r, color=LINE_COLOR, zorder=3))
    def arc(cx, cy, r, t1, t2): ax.add_patch(Arc((cx, cy), r*2, r*2, angle=0, theta1=t1, theta2=t2, color=LINE_COLOR, lw=lw, zorder=2))

    rect(0, 0, lm, am)
    line(lm/2, 0, lm/2, am)
    circle(lm/2, am/2, 9.15)
    dot(lm/2, am/2, 0.35)

    for side, x0 in [('L', 0), ('R', lm)]:
        sign = 1 if side == 'L' else -1
        rect(x0 if side == 'L' else x0 - 16.5, am/2 - 40.32/2, 16.5, 40.32)
        rect(x0 if side == 'L' else x0 - 5.5,  am/2 - 18.32/2, 5.5,  18.32)
        px_ = 11 if side == 'L' else lm - 11
        dot(px_, am/2, 0.35)
        a1, a2 = (-53, 53) if side == 'L' else (127, 233)
        arc(px_, am/2, 9.15, a1, a2)

    for cx, cy, t1, t2 in [(0, 0, 0, 90), (lm, 0, 90, 180), (0, am, 270, 360), (lm, am, 180, 270)]:
        arc(cx, cy, 1, t1, t2)

    gw = 7.32 / 2
    for xg, xd in [(0, -2), (lm, lm + 2)]:
        ax.add_patch(Rectangle((min(xg, xd), am/2 - gw), 2, 2*gw,
                               fill=True, facecolor='#aaa', edgecolor=LINE_COLOR, lw=lw, zorder=2))

    if limite_x is not None:
        ax.axvline(limite_x, color='#4488ff', lw=1.5, ls='--', alpha=0.5, zorder=4)


## 4 · Tracker global de campo — StrongKalmanTrackerV3
Re-ID cross-camera basado en posición en metros.

**Mejoras sobre KalmanFieldTracker:**
- **Asignación Hungarian** (`linear_sum_assignment`): óptima vs. greedy.
- **Two-stage matching**: detecciones HIGH_CONF → tracked; luego unmatched + LOW_CONF → lost. Recupera jugadores parcialmente tapados.
- **Buffer de lost** (2 s): los tracks no desaparecen inmediatamente, se mantienen como candidatos de re-ID.
- **Class-cap**: máximo 10+10+1+1+3 jugadores por clase (25 total). Evita IDs fantasma al saturar las clases.
- **Tracks tentativos**: un track no se emite hasta `MIN_HITS_CONFIRM` frames (filtro de detecciones puntuales).


In [16]:
# ═══════════════════════════════════════════════════════════
#  TRACKER GLOBAL DE CAMPO — StrongKalmanTrackerV3
#
#  Kalman 2D + Hungarian assignment + Two-stage matching
#  + Lost buffer + Class-cap
# ═══════════════════════════════════════════════════════════

from scipy.optimize import linear_sum_assignment

# ── Parámetros de Re-ID por color ──────────────────────────
GAP_COLOR_FRAMES     = 10    # frames sin detección antes de activar Re-ID color
COLOR_WEIGHT_M       = 3.0   # peso máximo del coste de color (m equivalentes)
MIN_HITS_CLASS       = 8     # hits mínimos para aplicar penalización de clase
MIN_CONF_NEW_UNKNOWN = 0.55  # conf mínima para crear ID con clase unknown

# ── Parámetros V3 ───────────────────────────────────────────
V3_HIGH_CONF  = 0.50   # umbral stage 1 (tracked)
V3_LOW_CONF   = 0.35   # umbral stage 2 (lost)
V3_LOST_S     = 2.0    # segundos en buffer lost
V3_GAP_HUE    = 5      # frames sin match para activar hue Re-ID
V3_HUE_W      = 4.0    # peso hue (m equivalentes)
V3_MAX_TOTAL  = 25     # cap total activos
V3_MAX_CLASS  = {'player_home':10, 'player_away':10,
                  'gk_home':1, 'gk_away':1, 'referee':3}
MIN_HITS_CONFIRM = 3   # frames hasta emitir un track como confirmado


def hue_sig_distance(sig1, sig2) -> float:
    a = np.sqrt(np.maximum(np.array(sig1, dtype=np.float32), 0))
    b = np.sqrt(np.maximum(np.array(sig2, dtype=np.float32), 0))
    return float(np.sqrt(np.sum((a - b) ** 2)) / np.sqrt(2.0))


class _KFTrackV2:
    def __init__(self, gid, pos_m, cls, conf, frame_idx,
                 color_desc=None, hue_sig=None, fps=30.0):
        self.gid           = gid
        self.conf          = conf
        self.last_frame    = frame_idx
        self.created       = frame_idx
        self.hits          = 1
        self.fps           = fps
        self.is_confirmed  = False
        self.class_history = defaultdict(int)
        self.class_history[cls] += 1
        self.cls    = cls
        self._alpha = 0.15
        self.color_desc = (np.array(color_desc, dtype=np.float32)
                           if color_desc is not None else None)
        self.hue_sig = (np.array(hue_sig, dtype=np.float32)
                        if hue_sig is not None else None)
        dt = 1.0
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.transitionMatrix    = np.array([[1,0,dt,0],[0,1,0,dt],
                                                 [0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.measurementMatrix   = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.processNoiseCov     = np.diag([0.05,0.05,0.5,0.5]).astype(np.float32)
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.3
        self.kf.errorCovPost        = np.eye(4, dtype=np.float32)
        self.kf.statePost           = np.array([[pos_m[0]],[pos_m[1]],
                                                 [0.],[0.]], np.float32)

    def predict(self):
        s = self.kf.predict()
        return np.array([float(s[0,0]), float(s[1,0])], np.float32)

    def correct(self, pos_m, cls, conf, frame_idx, color_desc=None, hue_sig=None):
        self.kf.correct(np.array([[pos_m[0]],[pos_m[1]]], np.float32))
        self.last_frame = frame_idx
        self.hits      += 1
        self.conf       = conf
        if self.hits >= MIN_HITS_CONFIRM:
            self.is_confirmed = True
        if cls != 'unknown':
            self.class_history[cls] += 1
            self.cls = max(self.class_history.items(), key=lambda kv: kv[1])[0]
        def _ema(cur, new):
            if new is None: return cur
            v = np.array(new, dtype=np.float32)
            return v if cur is None else (1 - self._alpha) * cur + self._alpha * v
        self.color_desc = _ema(self.color_desc, color_desc)
        self.hue_sig    = _ema(self.hue_sig, hue_sig)

    def hue_dist(self, hue_sig_det, w) -> float:
        if self.hue_sig is None or hue_sig_det is None: return 0.0
        return min(hue_sig_distance(self.hue_sig, hue_sig_det) * w, w)

    def color_dist_bgr(self, color_desc, w=COLOR_WEIGHT_M) -> float:
        if self.color_desc is None or color_desc is None: return 0.0
        raw = float(np.linalg.norm(
            self.color_desc - np.array(color_desc, dtype=np.float32)))
        return min(raw / 441.0 * w, w)

    @property
    def pos(self):
        return np.array([float(self.kf.statePost[0,0]),
                         float(self.kf.statePost[1,0])], np.float32)

    @property
    def speed_ms(self):
        vx = float(self.kf.statePost[2,0])
        vy = float(self.kf.statePost[3,0])
        return float(np.sqrt(vx**2 + vy**2)) * self.fps


class StrongKalmanTrackerV3:
    def __init__(self, fps=30.0, max_dist_m=6.0,
                 window_s=7.0, class_weight=8.0):
        self._next_id = 1
        self._tracked = {}   # gid -> _KFTrackV2
        self._lost    = {}   # gid -> _KFTrackV2
        self.fps      = fps
        self.max_dist = max_dist_m
        self.window   = int(fps * window_s)
        self.lost_win = int(fps * V3_LOST_S)
        self.class_w  = class_weight
        self.history  = defaultdict(list)

    def _cost_matrix(self, tracks_dict, dets, frame_idx):
        gids = list(tracks_dict.keys())
        C = np.full((len(gids), len(dets)), 1e6, dtype=np.float32)
        for i, gid in enumerate(gids):
            tr   = tracks_dict[gid]
            pred = tr.pos
            for j, d in enumerate(dets):
                pm = d.get('pos_m')
                if pm is None or None in pm: continue
                dist = float(np.linalg.norm(np.array(pm, np.float32) - pred))
                if dist > self.max_dist: continue
                pen  = 0.0
                dc   = d.get('cls', 'unknown')
                if dc != 'unknown' and dc != tr.cls and tr.hits >= MIN_HITS_CLASS:
                    pen += self.class_w
                gap = frame_idx - tr.last_frame
                if gap >= GAP_COLOR_FRAMES:
                    pen += tr.color_dist_bgr(d.get('color_desc'))
                if gap >= V3_GAP_HUE:
                    pen += tr.hue_dist(d.get('hue_sig'), V3_HUE_W)
                C[i, j] = dist + pen
        return gids, C

    def _match(self, tracks_dict, dets, frame_idx):
        if not tracks_dict or not dets:
            return [], list(tracks_dict.keys()), list(range(len(dets)))
        gids, C = self._cost_matrix(tracks_dict, dets, frame_idx)
        ri, ci  = linear_sum_assignment(C)
        matched, used_t, used_d = [], set(), set()
        for r, c in zip(ri, ci):
            if C[r, c] < 1e5:
                matched.append((gids[r], c)); used_t.add(r); used_d.add(c)
        um_g = [gids[i] for i in range(len(gids)) if i not in used_t]
        um_d = [j       for j in range(len(dets))  if j not in used_d]
        return matched, um_g, um_d

    def _class_count(self, cls):
        return sum(1 for tr in list(self._tracked.values()) + list(self._lost.values())
                   if tr.cls == cls)

    def _should_create(self, d):
        cls = d.get('cls', 'unknown')
        if len(self._tracked) + len(self._lost) >= V3_MAX_TOTAL:
            return False
        if cls in V3_MAX_CLASS and self._class_count(cls) >= V3_MAX_CLASS[cls]:
            return False
        if cls == 'unknown':
            return d.get('conf', 0.0) >= MIN_CONF_NEW_UNKNOWN
        return True

    def update(self, frame_idx, unified_dets):
        self._tracked = {g: t for g, t in self._tracked.items()
                         if frame_idx - t.last_frame <= self.window}
        self._lost    = {g: t for g, t in self._lost.items()
                         if frame_idx - t.last_frame <= self.lost_win}

        valid = [d for d in unified_dets
                 if d.get('pos_m') is not None and None not in d['pos_m']]
        high  = [d for d in valid if d.get('conf', 0) >= V3_HIGH_CONF]
        low   = [d for d in valid if V3_LOW_CONF <= d.get('conf', 0) < V3_HIGH_CONF]

        for tr in self._tracked.values(): tr.predict()
        for tr in self._lost.values():    tr.predict()

        # Stage 1: high-conf <-> tracked
        m1, um_t1, um_h1 = self._match(self._tracked, high, frame_idx)
        for gid, di in m1:
            d = high[di]
            self._tracked[gid].correct(d['pos_m'], d.get('cls','unknown'),
                                       d.get('conf',1.0), frame_idx,
                                       color_desc=d.get('color_desc'),
                                       hue_sig=d.get('hue_sig'))
        for gid in um_t1:
            self._lost[gid] = self._tracked.pop(gid)

        # Stage 2: unmatched high + low <-> lost
        s2   = [high[j] for j in um_h1] + low
        n_hi = len(um_h1)
        m2, _, um_s2 = self._match(self._lost, s2, frame_idx)
        for gid, di in m2:
            d = s2[di]
            self._lost[gid].correct(d['pos_m'], d.get('cls','unknown'),
                                    d.get('conf',1.0), frame_idx,
                                    color_desc=d.get('color_desc'),
                                    hue_sig=d.get('hue_sig'))
            self._tracked[gid] = self._lost.pop(gid)

        # Nuevas pistas sólo desde unmatched high-conf
        for di in um_s2:
            if di < n_hi:
                d = s2[di]
                if self._should_create(d):
                    gid = self._next_id; self._next_id += 1
                    tr  = _KFTrackV2(gid, d['pos_m'], d.get('cls','unknown'),
                                     d.get('conf',1.0), frame_idx,
                                     color_desc=d.get('color_desc'),
                                     hue_sig=d.get('hue_sig'), fps=self.fps)
                    self._tracked[gid] = tr

        result = []
        for gid, tr in self._tracked.items():
            if not tr.is_confirmed: continue
            self.history[gid].append(
                (frame_idx, float(tr.pos[0]), float(tr.pos[1]), tr.cls,
                 round(tr.speed_ms, 2)))
            result.append({'gid': gid, 'cls': tr.cls, 'speed_ms': tr.speed_ms,
                           'pos_m': tuple(tr.pos), 'conf': tr.conf,
                           'cam': '?'})
        return result

    def active_ids(self):
        return set(self._tracked.keys())

    def get_trajectory(self, gid):
        return self.history.get(gid, [])


print('StrongKalmanTrackerV3 definido')
print(f'  two-stage HIGH={V3_HIGH_CONF} LOW={V3_LOW_CONF}  lost={V3_LOST_S}s')
print(f'  class-cap: {V3_MAX_CLASS}  total<={V3_MAX_TOTAL}')


StrongKalmanTrackerV3 definido
  two-stage HIGH=0.5 LOW=0.35  lost=2.0s
  class-cap: {'player_home': 10, 'player_away': 10, 'gk_home': 1, 'gk_away': 1, 'referee': 3}  total<=25


## 5 · Carga de modelos y recursos
2 instancias YOLO independientes (una por cámara), homografías y PKLs.

In [17]:
# ═══════════════════════════════════════════════════════════
#  CARGA DE MODELOS, HOMOGRAFIAS Y PKLs
# ═══════════════════════════════════════════════════════════

# --- Modelos YOLO (2 instancias independientes: una por camara) ----------
model_A = YOLO(MODEL_PLAYERS)
model_B = YOLO(MODEL_PLAYERS)
print(f"Modelos YOLO cargados: {MODEL_PLAYERS}")

# --- Homografias -----------------------------------------------------------
H_a = np.load(H_A_PATH)   # Cam A (mitad superior)
H_b = np.load(H_B_PATH)   # Cam B (mitad inferior)
print(f"Homografias cargadas:  H_a {H_a.shape}  H_b {H_b.shape}")

# --- PKLs de color (dual dia/noche con fallback) --------------------------
protos_day   = pickle.load(open(PROTO_DAY_PATH,   'rb')) if os.path.exists(PROTO_DAY_PATH)   else None
protos_night = pickle.load(open(PROTO_NIGHT_PATH, 'rb')) if os.path.exists(PROTO_NIGHT_PATH) else None

if protos_day is None and protos_night is None:
    if os.path.exists(PROTO_PATH):
        protos_day = protos_night = pickle.load(open(PROTO_PATH, 'rb'))
        print(f"AVISO: usando PKL legacy ({PROTO_PATH})")
    else:
        raise FileNotFoundError(
            "No se encontro ningun PKL. Ejecuta generar_prototipos_dual.py"
        )
elif protos_day   is None: print("AVISO: PKL dia no encontrado -> fallback noche")
elif protos_night is None: print("AVISO: PKL noche no encontrado -> fallback dia")

# Placeholder hasta leer el primer frame
protos = protos_day or protos_night

# Resumen
print(f"\n{'PKL':<12}  {'Estado':<5}  {'Clases'}  Ruta")
print("─" * 55)
for lbl, pk, path in [('PKL dia', protos_day, PROTO_DAY_PATH),
                       ('PKL noche', protos_night, PROTO_NIGHT_PATH)]:
    if pk:
        ns = '  '.join(f"{c[:3]}:{pk[c]['n_samples']}" for c in pk)
        print(f"{lbl:<12}  OK     {len(pk):<5}  {path}")
        print(f"{'':12}         {ns}")
    else:
        print(f"{lbl:<12}  --            {path}")

# --- Video info -----------------------------------------------------------
cap_info = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(cap_info.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap_info.get(cv2.CAP_PROP_FPS)
VID_W        = int(cap_info.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H        = int(cap_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_info.release()

HALF_H   = VID_H // 2   # altura de cada camara (frame completo dividido por 2)
END_FRAME = min(START_FRAME + N_FRAMES, TOTAL_FRAMES)

# Limite X en metros entre zonas A y B (para visualizacion)
# Se calcula proyectando las esquinas de ambas camaras
corners_a = esquinas_metros(H_a, VID_W, HALF_H)
corners_b = esquinas_metros(H_b, VID_W, HALF_H)
LIMITE_X  = float(np.mean([corners_a[:, 0].max(), corners_b[:, 0].min()]))

print(f"\nVideo:  {VID_W}x{VID_H}  |  {TOTAL_FRAMES} frames  |  {FPS:.1f} fps")
print(f"Camara: mitad superior ({HALF_H}px) = Cam A  |  mitad inferior = Cam B")
print(f"Segmento: frames {START_FRAME}–{END_FRAME}  ({N_FRAMES/FPS:.1f} s)")
print(f"Limite X campo (A/B): {LIMITE_X:.1f} m")
print(f"Opciones: CLASSIFY_MAX_SCORE={CLASSIFY_MAX_SCORE}  SEAM_FUSION={SEAM_FUSION_M}m")


Modelos YOLO cargados: runs/detect/modelo_players_v24_panoramic2/weights/best.pt
Homografias cargadas:  H_a (3, 3)  H_b (3, 3)

PKL           Estado  Clases  Ruta
───────────────────────────────────────────────────────
PKL dia       OK     5      prototypes_v3_day.pkl
                     pla:20  pla:20  ref:20  gk_:20  gk_:20
PKL noche     OK     4      prototypes_v3_night.pkl
                     pla:20  pla:20  ref:8  gk_:16

Video:  2048x2048  |  215710 frames  |  30.0 fps
Camara: mitad superior (1024px) = Cam A  |  mitad inferior = Cam B
Segmento: frames 18000–18900  (30.0 s)
Limite X campo (A/B): 53.5 m
Opciones: CLASSIFY_MAX_SCORE=80.0  SEAM_FUSION=2.5m


## Preprocesado: alpha=1+crop
Pre-computa los mapas de undistort para ambas cámaras (se hace una sola vez).
`preprocess_half(half_bgr, cam)` aplica elastic undo + undistort + crop + resize.

In [18]:
# ═══════════════════════════════════════════════════════════
#  PRE-CÓMPUTO DE MAPAS DE UNDISTORT (una sola vez)
#  alpha=1 → conserva todos los píxeles (bordes negros)
#  crop    → recorta al ROI válido y redimensiona al tamaño original
# ═══════════════════════════════════════════════════════════

_h_half, _w_half = HALF_H, VID_W

_new_K_a, _roi_a = cv2.getOptimalNewCameraMatrix(
    K_LEFT,  DIST_COEFFS, (_w_half, _h_half), alpha=ALPHA_UNDIST)
_map1_a, _map2_a = cv2.initUndistortRectifyMap(
    K_LEFT,  DIST_COEFFS, None, _new_K_a, (_w_half, _h_half), cv2.CV_32FC1)

_new_K_b, _roi_b = cv2.getOptimalNewCameraMatrix(
    K_RIGHT, DIST_COEFFS, (_w_half, _h_half), alpha=ALPHA_UNDIST)
_map1_b, _map2_b = cv2.initUndistortRectifyMap(
    K_RIGHT, DIST_COEFFS, None, _new_K_b, (_w_half, _h_half), cv2.CV_32FC1)

print(f"Mapas de undistort pre-computados (alpha={ALPHA_UNDIST})")
print(f"  ROI cam A: {_roi_a}")
print(f"  ROI cam B: {_roi_b}")


def preprocess_half(half_bgr, cam='left'):
    """
    Aplica alpha=1+crop a una mitad del frame VEO.
    Resultado: misma resolución que la entrada, sin bordes negros,
    listo para YOLO con la homografía alpha=1+crop.
    """
    # 1. Elastic undo (corrige compresión elástica VEO)
    prep = undo_elastic_compression(half_bgr)
    # 2. Undistort con alpha=1
    if cam == 'left':
        undist = cv2.remap(prep, _map1_a, _map2_a, cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT)
        roi = _roi_a
    else:
        undist = cv2.remap(prep, _map1_b, _map2_b, cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_CONSTANT)
        roi = _roi_b
    # 3. Crop al ROI válido + resize al tamaño original
    h_o, w_o = undist.shape[:2]
    x, y, rw, rh = roi
    crop = undist[y:y+rh, x:x+rw]
    return cv2.resize(crop, (w_o, h_o), interpolation=cv2.INTER_LINEAR)


# Verificacion visual: preview del preprocesado sobre el frame de inicio
cap_chk = cv2.VideoCapture(VIDEO_PATH)
cap_chk.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
_, _f_chk = cap_chk.read()
cap_chk.release()

_a_raw = _f_chk[:HALF_H, :]
_b_raw = _f_chk[HALF_H:,  :]
_a_proc = preprocess_half(_a_raw, 'left')
_b_proc = preprocess_half(_b_raw, 'right')

fig_pre, ax_pre = plt.subplots(2, 2, figsize=(22, 8))
fig_pre.patch.set_facecolor(DARK_BG)
for ax, img, title in [
    (ax_pre[0,0], _a_raw,  'Cam A — RAW'),
    (ax_pre[0,1], _a_proc, 'Cam A — alpha=1+crop'),
    (ax_pre[1,0], _b_raw,  'Cam B — RAW'),
    (ax_pre[1,1], _b_proc, 'Cam B — alpha=1+crop'),
]:
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')
fig_pre.suptitle(f'Preprocesado alpha={ALPHA_UNDIST}+crop — frame {START_FRAME}',
                 color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


Mapas de undistort pre-computados (alpha=1.0)
  ROI cam A: (141, 172, 1789, 692)
  ROI cam B: (140, 171, 1788, 696)


C:\Users\xavia\AppData\Local\Temp\ipykernel_28828\529334048.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Preview del segmento
Verifica que el frame de inicio es correcto y que la luz se detecta bien.

In [19]:
# ═══════════════════════════════════════════════════════════
#  PREVIEW DEL SEGMENTO A ANALIZAR
#  Muestra el frame inicial con las 2 camaras y el mapa 2D
# ═══════════════════════════════════════════════════════════

cap_prev = cv2.VideoCapture(VIDEO_PATH)
cap_prev.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
ret, frame_prev = cap_prev.read()
cap_prev.release()

if not ret:
    print(f"No se pudo leer el frame {START_FRAME}")
else:
    cam_a_prev = preprocess_half(frame_prev[:HALF_H, :], 'left')
    cam_b_prev = preprocess_half(frame_prev[HALF_H:, :], 'right')

    brightness = frame_brightness(frame_prev)
    cond       = 'DAY' if brightness > BRIGHT_THRESH else 'NIGHT'
    protos_sel = select_protos(frame_prev, protos_day, protos_night)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5),
                              gridspec_kw={'width_ratios': [2, 2, 1.5]})
    fig.patch.set_facecolor(DARK_BG)

    for ax, img, title in [
        (axes[0], cam_a_prev, 'Cam A — mitad superior'),
        (axes[1], cam_b_prev, 'Cam B — mitad inferior'),
    ]:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, color='white', fontsize=10)
        ax.axis('off')

    draw_pitch_mpl(axes[2], L_M, A_M, LIMITE_X)
    axes[2].set_title('Campo 2D', color='white', fontsize=10)

    fig.suptitle(
        f"Frame {START_FRAME}  |  brillo={brightness:.1f}  |  condicion={cond}  |  PKL={cond.lower()}",
        color='white', fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    print(f"Segmento: {START_FRAME} → {END_FRAME}  ({N_FRAMES} frames, {N_FRAMES/FPS:.1f} s)")
    print(f"Condicion de luz detectada: {cond}  (brillo={brightness:.1f}, umbral={BRIGHT_THRESH})")


Segmento: 18000 → 18900  (900 frames, 30.0 s)
Condicion de luz detectada: DAY  (brillo=101.6, umbral=75.0)


C:\Users\xavia\AppData\Local\Temp\ipykernel_28828\354262826.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Procesado del segmento
ByteTrack (x2 cámaras) → **seam deduplication** → `StrongKalmanTrackerV3` → heatmaps.

**Qué hace cada paso:**
1. ByteTrack independiente en Cam A y Cam B.
2. `_unify_seam()`: fusiona duplicados en la franja A/B (greedy por distancia en metros).
3. `StrongKalmanTrackerV3.update()`: asigna IDs globales estables con two-stage matching y class-cap.
4. Ball detection con Kalman (si `MODEL_BALL` existe).

> ⏱️ A 30 fps: 900 frames ≈ 30 s de video ≈ 2-4 min de cómputo.


In [20]:
# ═══════════════════════════════════════════════════════════
#  BUCLE PRINCIPAL DE PROCESADO
#  ByteTrack x2 camaras → seam dedup → KalmanFieldTracker
#  Balón con Kalman · Trayectorias · Heatmaps · Stats
# ═══════════════════════════════════════════════════════════

# ---------- Inicializacion --------------------------------------------------

tracker = StrongKalmanTrackerV3(fps=FPS,
                                 max_dist_m=TRACK_MAX_DIST_M,
                                 window_s=TRACK_WINDOW_S)

# Heatmaps: cuadricula por clase (campo en metros, resolución ESCALA px/m)
HM_W = int(L_M * ESCALA)
HM_H = int(A_M * ESCALA)
heatmaps = {cls: np.zeros((HM_H, HM_W), dtype=np.float32)
            for cls in CLASSES_ORDER}
heatmap_ball = np.zeros((HM_H, HM_W), dtype=np.float32)

# Registros tabulares
frame_records = []   # [{frame, gid, cls, x_m, y_m, conf, speed_ms, cam}]
ball_records  = []   # [{frame, x_m, y_m, conf}]

# ── Seam deduplication ────────────────────────────────────────────────────
def _unify_seam(dets_a, dets_b, cut_x, margin=SEAM_MARGIN_M,
                fusion_thr=SEAM_FUSION_M):
    """
    Fusiona detecciones de Cam A y Cam B eliminando duplicados
    en la franja de solapamiento (±margin metros alrededor de cut_x).

    - Detecciones fuera de la zona de solapamiento: pasan directamente.
    - En la zona: asignacion greedy por distancia en metros.
      Si d < fusion_thr → se promedian las posiciones y se queda
      la clase con mayor confianza.
    - Compatibilidad de claves: adapta dict de _track_half
      ('pos_m', 'cls', 'conf') al formato de unify_cameras.
    """
    from scipy.spatial.distance import cdist as _cdist

    finales = []

    # Zonas sin solapamiento
    zone_a_only = [d for d in dets_a
                   if d['pos_m'] is not None and d['pos_m'][0] < cut_x - margin]
    zone_b_only = [d for d in dets_b
                   if d['pos_m'] is not None and d['pos_m'][0] > cut_x + margin]
    finales.extend(zone_a_only)
    finales.extend(zone_b_only)

    # Zona de solapamiento
    front_a = [d for d in dets_a
               if d['pos_m'] is not None
               and abs(d['pos_m'][0] - cut_x) <= margin]
    front_b = [d for d in dets_b
               if d['pos_m'] is not None
               and abs(d['pos_m'][0] - cut_x) <= margin]

    if not front_a and not front_b:
        return finales

    if not front_a:
        finales.extend(front_b)
        return finales
    if not front_b:
        finales.extend(front_a)
        return finales

    pts_a = np.array([d['pos_m'] for d in front_a], dtype=np.float32)
    pts_b = np.array([d['pos_m'] for d in front_b], dtype=np.float32)
    D     = _cdist(pts_a, pts_b)

    used_a, used_b = set(), set()
    while True:
        mask  = np.ones_like(D, dtype=bool)
        for i in used_a: mask[i, :] = False
        for j in used_b: mask[:, j] = False
        if not mask.any():
            break
        d_min = D[mask].min()
        if d_min >= fusion_thr:
            break
        coords = np.argwhere((D == d_min) & mask)
        i, j   = int(coords[0, 0]), int(coords[0, 1])
        used_a.add(i); used_b.add(j)
        da, db = front_a[i], front_b[j]
        # Fusion: media de posiciones, clase de mayor conf, conf maxima
        px = (da['pos_m'][0] + db['pos_m'][0]) / 2.0
        py = (da['pos_m'][1] + db['pos_m'][1]) / 2.0
        merged = dict(da)
        merged['pos_m'] = (px, py)
        merged['cam']   = 'AB'
        if da['cls'] != db['cls']:
            merged['cls']  = da['cls'] if da['conf'] >= db['conf'] else db['cls']
        merged['conf'] = max(da['conf'], db['conf'])
        finales.append(merged)

    # Residuos: los que no se emparejaron
    for i, d in enumerate(front_a):
        if i not in used_a:
            finales.append(d)
    for j, d in enumerate(front_b):
        if j not in used_b:
            finales.append(d)

    return finales


# ── NMS global post-unificacion ───────────────────────────────────────────
def _global_nms(dets, min_dist=GLOBAL_NMS_M, max_dets=MAX_DETS_FRAME):
    """
    Elimina detecciones duplicadas que escaparon al seam dedup.
    Algoritmo: ordena por confianza descendente; descarta cualquier det
    que este a menos de min_dist metros de una ya aceptada.
    Finalmente limita a max_dets (= 26 = 11+11+ref+3 margen).

    Razon para 35-68 IDs: YOLO detecta espectadores/banquillo/fotografos
    cerca de las lineas de banda, y la homografia los proyecta dentro
    del area de campo extendida (-5m, L_M+5m). Con filtro estricto
    (0..L_M, 0..A_M) + NMS + cap → bajamos a 22-26 por frame.
    """
    # Solo dets con posicion valida DENTRO del campo (sin margen extra)
    valid = [d for d in dets
             if d['pos_m'] is not None
             and 0.0 <= d['pos_m'][0] <= L_M
             and 0.0 <= d['pos_m'][1] <= A_M]
    # Ordenar por confianza descendente
    valid = sorted(valid, key=lambda d: -d['conf'])

    kept, kept_pos = [], []
    for d in valid:
        p = np.array(d['pos_m'], dtype=np.float32)
        too_close = any(np.linalg.norm(p - q) < min_dist for q in kept_pos)
        if not too_close:
            kept.append(d)
            kept_pos.append(p)
        if len(kept) >= max_dets:
            break
    return kept


# ── Deteccion en una mitad del frame ─────────────────────────────────────
def _track_half(model, half_img, H, conf=CONF_THRESH):
    """ByteTrack en una mitad; devuelve lista de dets con pos en metros.
    Ahora incluye 'color_desc' (descriptor BGR medio del torso) para
    Re-ID por apariencia en KalmanFieldTracker v2.
    """
    results = model.track(half_img, persist=True, tracker='bytetrack.yaml',
                          conf=conf, verbose=False)
    dets = []
    if results and results[0].boxes is not None:
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
            conf_val = float(box.conf[0])
            crop = half_img[max(0,int(y1)):int(y2), max(0,int(x1)):int(x2)]
            if crop.size == 0:
                continue
            torso    = get_torso_crop(crop)
            cls_name, _ = classify_player(torso, protos)

            # Descriptor de color para Re-ID (media BGR del torso filtrada)
            _, color_mean, _ = get_player_color(torso)

            cx_px = (x1 + x2) / 2.0
            cy_px = float(y2)
            try:
                xm, ym = pixel_to_field(cx_px, cy_px, H)
            except Exception:
                xm, ym = None, None
            # Si unknown, intentar reclasificar como portero (posicion + color)
            if cls_name == 'unknown' and xm is not None:
                _det_tmp = {'pos_m': (xm, ym), 'color_desc': color_mean.tolist()}
                _gk_cls = classify_gk_score(
                    _det_tmp,
                    protos.get('gk_home'), protos.get('gk_away'),
                    home_attacks_left=HOME_ATTACKS_LEFT,
                )
                if _gk_cls is not None:
                    cls_name = _gk_cls
            dets.append({
                'bbox_px':    (int(x1), int(y1), int(x2), int(y2)),
                'conf':       conf_val,
                'cls':        cls_name,
                'pos_m':      (xm, ym) if xm is not None else None,
                'color_desc': color_mean.tolist(),
            })
    return dets


# ── Deteccion de balon ────────────────────────────────────────────────────
_ball_model = None
_ball_available = False
if os.path.exists(MODEL_BALL):
    try:
        _ball_model   = YOLO(MODEL_BALL)
        _ball_available = True
        print(f"Modelo de balon cargado: {MODEL_BALL}")
    except Exception as _e:
        print(f"AVISO: no se pudo cargar modelo balon ({_e})")
else:
    print(f"AVISO: modelo de balon no encontrado ({MODEL_BALL}) — se omite")

# Kalman para el balon (1D en cada coordenada)
_ball_kf = cv2.KalmanFilter(4, 2)
_ball_kf.transitionMatrix = np.array(
    [[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], dtype=np.float32)
_ball_kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], dtype=np.float32)
_ball_kf.processNoiseCov   = np.diag([0.1, 0.1, 1.0, 1.0]).astype(np.float32)
_ball_kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5
_ball_kf.errorCovPost = np.eye(4, dtype=np.float32)
_ball_initialized = False


def _detect_ball(frame, H_a, H_b, conf=CONF_BALL):
    """Detecta el balon en ambas mitades; devuelve (xm, ym, conf) o None."""
    global _ball_initialized
    if not _ball_available:
        return None

    best_det = None
    for half, H, cam_side in [
        (preprocess_half(frame[:HALF_H, :], 'left'),  H_a, 'left'),
        (preprocess_half(frame[HALF_H:,  :], 'right'), H_b, 'right'),
    ]:
        res = _ball_model.predict(half, conf=conf, verbose=False)
        if not res or res[0].boxes is None:
            continue
        for box in res[0].boxes:
            bc = float(box.conf[0])
            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
            cx  = (x1 + x2) / 2.0
            cy  = (y1 + y2) / 2.0
            try:
                xm, ym = pixel_to_field(cx, cy, H)
                if not (-5 <= xm <= L_M + 5 and -5 <= ym <= A_M + 5):
                    continue
            except Exception:
                continue
            if best_det is None or bc > best_det[2]:
                best_det = (xm, ym, bc)

    if best_det is None:
        # Predecir con Kalman si ya esta inicializado
        if _ball_initialized:
            s = _ball_kf.predict()
            return (float(s[0, 0]), float(s[1, 0]), 0.0)
        return None

    # Corregir Kalman
    if not _ball_initialized:
        _ball_kf.statePost = np.array(
            [[best_det[0]], [best_det[1]], [0.0], [0.0]], dtype=np.float32)
        _ball_initialized = True
    else:
        _ball_kf.predict()
    meas = np.array([[best_det[0]], [best_det[1]]], dtype=np.float32)
    _ball_kf.correct(meas)
    s = _ball_kf.statePost
    return (float(s[0, 0]), float(s[1, 0]), best_det[2])


# ── Bucle principal ───────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

n_processed = 0
print(f"Procesando {N_FRAMES} frames [{START_FRAME}–{END_FRAME}]…")
print(f"  Balon: {'SI' if _ball_available else 'NO (modelo no encontrado)'}  |  "
      f"Seam fusion: {SEAM_FUSION_M} m  |  Classify threshold: {CLASSIFY_MAX_SCORE}")

while n_processed < N_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx = START_FRAME + n_processed
    cam_a = preprocess_half(frame[:HALF_H, :], 'left')
    cam_b = preprocess_half(frame[HALF_H:,  :], 'right')

    # Actualizar PKL segun iluminacion del frame actual
    protos = select_protos(frame, protos_day, protos_night)

    # Detectar + ByteTrack en cada mitad
    dets_a = _track_half(model_A, cam_a, H_a)
    dets_b = _track_half(model_B, cam_b, H_b)

    # Fusionar con deduplicacion en la costura
    unified = _unify_seam(
        [{**d, 'cam': 'A'} for d in dets_a],
        [{**d, 'cam': 'B'} for d in dets_b],
        cut_x=LIMITE_X,
    )

    # NMS global: elimina duplicados residuales + cap de jugadores
    # (filtra espectadores/banquillo proyectados dentro del campo)
    unified = _global_nms(unified)

    # Re-ID global con Kalman
    tracked = tracker.update(frame_idx, unified)

    # Balón
    ball_det = _detect_ball(frame, H_a, H_b)

    # Acumular heatmaps y registros
    for d in tracked:
        if d['pos_m'] is None:
            continue
        xm, ym = d['pos_m']
        if not (0 <= xm <= L_M and 0 <= ym <= A_M):
            continue
        cls = d['cls']
        px  = int(np.clip(xm * ESCALA, 0, HM_W - 1))
        py  = int(np.clip(ym * ESCALA, 0, HM_H - 1))
        if cls in heatmaps:
            heatmaps[cls][py, px] += 1.0
        frame_records.append({
            'frame':    frame_idx,
            'gid':      d['gid'],
            'cls':      cls,
            'x_m':      round(xm, 2),
            'y_m':      round(ym, 2),
            'conf':     round(d['conf'], 3),
            'speed_ms': round(d.get('speed_ms', 0.0), 2),
            'cam':      d.get('cam', '?'),
        })

    if ball_det is not None:
        bxm, bym, bconf = ball_det
        if 0 <= bxm <= L_M and 0 <= bym <= A_M:
            bpx = int(np.clip(bxm * ESCALA, 0, HM_W - 1))
            bpy = int(np.clip(bym * ESCALA, 0, HM_H - 1))
            heatmap_ball[bpy, bpx] += 1.0
            ball_records.append({
                'frame': frame_idx,
                'x_m':   round(bxm, 2),
                'y_m':   round(bym, 2),
                'conf':  round(bconf, 3),
            })

    n_processed += 1
    if n_processed % 150 == 0 or n_processed == N_FRAMES:
        pct    = 100 * n_processed / N_FRAMES
        unkn   = sum(1 for r in frame_records if r['frame'] == frame_idx
                     and r['cls'] == 'unknown')
        print(f"  [{pct:5.1f}%]  frame {frame_idx}  |  "
              f"IDs activos: {len(tracker.active_ids())}  |  "
              f"unknown: {unkn}  |  registros: {len(frame_records)}")

cap.release()

# ── DataFrame ─────────────────────────────────────────────────────────────
df      = pd.DataFrame(frame_records)
df_ball = pd.DataFrame(ball_records)

if len(df):
    df['time_s'] = (df['frame'] - START_FRAME) / FPS
if len(df_ball):
    df_ball['time_s'] = (df_ball['frame'] - START_FRAME) / FPS

print(f"\nProcesado completado.")
print(f"  Frames procesados : {n_processed}")
print(f"  Registros jugadores: {len(df)}  |  IDs unicos: "
      f"{df['gid'].nunique() if len(df) else 0}")
print(f"  Registros balon   : {len(df_ball)}")
if len(df):
    cls_counts = df.groupby('cls').size()
    print(f"  Distribucion de clases:\n{cls_counts.to_string()}")
    unk_pct = 100 * cls_counts.get('unknown', 0) / len(df)
    print(f"  Detecciones 'unknown': {unk_pct:.1f}%  "
          f"(ideal < 5%; si > 15%, revisar PKL o CLASSIFY_MAX_SCORE)")


Modelo de balon cargado: runs/detect/modelo_ball_v33/weights/best.pt
Procesando 900 frames [18000–18900]…
  Balon: SI  |  Seam fusion: 2.5 m  |  Classify threshold: 80.0
  [ 16.7%]  frame 18149  |  IDs activos: 12  |  unknown: 0  |  registros: 2082
  [ 33.3%]  frame 18299  |  IDs activos: 17  |  unknown: 0  |  registros: 4512
  [ 50.0%]  frame 18449  |  IDs activos: 16  |  unknown: 0  |  registros: 7105
  [ 66.7%]  frame 18599  |  IDs activos: 16  |  unknown: 0  |  registros: 9642
  [ 83.3%]  frame 18749  |  IDs activos: 17  |  unknown: 0  |  registros: 11872
  [100.0%]  frame 18899  |  IDs activos: 15  |  unknown: 0  |  registros: 14186

Procesado completado.
  Frames procesados : 900
  Registros jugadores: 14186  |  IDs unicos: 50
  Registros balon   : 66
  Distribucion de clases:
cls
gk_away         352
gk_home         936
player_away    6084
player_home    6622
referee         186
unknown           6
  Detecciones 'unknown': 0.0%  (ideal < 5%; si > 15%, revisar PKL o CLASSIFY_MAX_S

## 8 · Visor interactivo de frames
Navega por el segmento frame a frame, con IDs, bboxes y mapa táctico en tiempo real.

In [21]:
# ═══════════════════════════════════════════════════════════
#  VISOR INTERACTIVO DE FRAME
#  Muestra frame + detecciones + IDs + mapa tactico 2D
# ═══════════════════════════════════════════════════════════

# ── Widgets ────────────────────────────────────────────────────────────────
slider_frame = widgets.IntSlider(
    value=START_FRAME, min=START_FRAME, max=END_FRAME - 1, step=1,
    description='Frame:', continuous_update=False,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '60px'}
)
chk_ids      = widgets.Checkbox(value=True,  description='Mostrar IDs',     indent=False)
chk_bbox     = widgets.Checkbox(value=True,  description='Mostrar bboxes',  indent=False)
chk_map      = widgets.Checkbox(value=True,  description='Mapa tactico',    indent=False)
chk_traj     = widgets.Checkbox(value=False, description='Trayectorias',    indent=False)
TRAJ_FRAMES  = widgets.IntSlider(value=60, min=10, max=300, step=10,
                                  description='Traj (f):', continuous_update=False,
                                  style={'description_width': '70px'})

out_viewer = widgets.Output()

def _render_viewer(frame_idx, show_ids, show_bbox, show_map, show_traj, traj_len):
    # Leer frame del disco
    cap_v = cv2.VideoCapture(VIDEO_PATH)
    cap_v.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame_v = cap_v.read()
    cap_v.release()
    if not ret:
        print(f"No se pudo leer frame {frame_idx}")
        return

    cam_a_v = frame_v[:HALF_H, :].copy()
    cam_b_v = frame_v[HALF_H:,  :].copy()
    cam_a_v = preprocess_half(cam_a_v, 'left')
    cam_b_v = preprocess_half(cam_b_v, 'right')

    # Detecciones del frame en el DataFrame
    df_f = df[df['frame'] == frame_idx] if len(df) else pd.DataFrame()

    # Dibujar sobre cada mitad
    for _, row in df_f.iterrows():
        cam_img = cam_a_v if row['cam'] == 'A' else cam_b_v
        # Recuperar bbox del tracker history (usamos registro)
        # Nota: guardamos en el DataFrame solo posicion, no bbox px.
        # Re-ejecutamos deteccion ligera solo para la visualizacion:
        # Para evitar re-YOLO, almacenamos bbox en frame_records durante c7.
        # (ver nota al pie)
        pass

    # Reconstruir bbox del history para este frame
    # Las bbox_px NO se guardan en el DataFrame (solo pos_m).
    # Ejecutamos YOLO inference (sin tracker, rapido) solo para preview.
    _protos_v = select_protos(frame_v, protos_day, protos_night)

    def _annotate(half, H, model_v):
        res = model_v.predict(half, conf=CONF_THRESH, verbose=False)
        if not res or res[0].boxes is None:
            return
        boxes = res[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
            crop = half[max(0,y1):y2, max(0,x1):x2]
            if crop.size == 0:
                continue
            cls_name, _ = classify_player(get_torso_crop(crop), _protos_v)
            color = COLOR_MAP_BGR.get(cls_name, (200, 200, 200))
            if show_bbox:
                cv2.rectangle(half, (x1, y1), (x2, y2), color, 2)
            # Buscar gid en df para este frame/posicion
            cx_px = (x1 + x2) / 2.0; cy_px = float(y2)
            try:
                xm, ym = pixel_to_field(cx_px, cy_px, H)
            except Exception:
                xm, ym = None, None
            if show_ids and xm is not None and len(df_f):
                # Buscar el gid mas cercano en campo
                mask = (df_f['cam'] == ('A' if H is H_a else 'B'))
                sub  = df_f[mask]
                if len(sub):
                    dists = np.sqrt((sub['x_m'] - xm)**2 + (sub['y_m'] - ym)**2)
                    best  = sub.iloc[dists.argmin()]
                    if dists.min() < 3.0:
                        label = f"#{int(best['gid'])}"
                        cv2.putText(half, label, (x1, y1 - 5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2,
                                    cv2.LINE_AA)

    _annotate(cam_a_v, H_a, model_A)
    _annotate(cam_b_v, H_b, model_B)

    # ── Figura ────────────────────────────────────────────────────────────
    ncols = 3 if show_map else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
    fig.patch.set_facecolor(DARK_BG)

    for ax, img, title in zip(axes[:2],
                               [cam_a_v, cam_b_v],
                               ['Cam A', 'Cam B']):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, color='white', fontsize=9)
        ax.axis('off')

    if show_map:
        ax_map = axes[2]
        draw_pitch_mpl(ax_map, L_M, A_M, LIMITE_X)

        # Trayectorias
        if show_traj and len(df):
            df_traj = df[(df['frame'] >= frame_idx - traj_len) &
                         (df['frame'] <= frame_idx)]
            for gid, grp in df_traj.groupby('gid'):
                grp = grp.sort_values('frame')
                if len(grp) < 2:
                    continue
                cls_t = grp.iloc[-1]['cls']
                hex_c = COLOR_MAP_HEX.get(cls_t, '#aaaaaa')
                ax_map.plot(grp['x_m'], grp['y_m'],
                            color=hex_c, alpha=0.4, lw=1.5, zorder=3)

        # Posiciones actuales
        for _, row in df_f.iterrows():
            hex_c = COLOR_MAP_HEX.get(row['cls'], '#aaaaaa')
            ax_map.scatter(row['x_m'], row['y_m'],
                           color=hex_c, s=90, zorder=5,
                           edgecolors='white', linewidths=0.8)
            if show_ids:
                ax_map.text(row['x_m'], row['y_m'] - 1.5,
                            f"#{int(row['gid'])}", color='white',
                            fontsize=7, ha='center', zorder=6,
                            fontweight='bold')

        ax_map.set_title('Mapa tactico', color='white', fontsize=9)

    # Leyenda
    handles = [mpatches.Patch(color=COLOR_MAP_HEX.get(c, '#aaa'), label=c)
               for c in CLASSES_ORDER]
    axes[0].legend(handles=handles, loc='lower left', fontsize=7,
                   facecolor='#1a1a2e', labelcolor='white',
                   edgecolor='#333', framealpha=0.8, ncol=2)

    t_s = (frame_idx - START_FRAME) / FPS
    fig.suptitle(f"Frame {frame_idx}  |  t={t_s:.1f} s", color='white',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    with out_viewer:
        out_viewer.clear_output(wait=True)
        display(fig)
        plt.close(fig)


# ── Observadores ───────────────────────────────────────────────────────────
def _on_change(_):
    _render_viewer(slider_frame.value, chk_ids.value, chk_bbox.value,
                   chk_map.value, chk_traj.value, TRAJ_FRAMES.value)

for w in [slider_frame, chk_ids, chk_bbox, chk_map, chk_traj, TRAJ_FRAMES]:
    w.observe(_on_change, names='value')

# ── Layout y primer render ──────────────────────────────────────────────────
controls = widgets.VBox([
    slider_frame,
    widgets.HBox([chk_ids, chk_bbox, chk_map, chk_traj, TRAJ_FRAMES]),
])
display(widgets.VBox([controls, out_viewer]))
_render_viewer(START_FRAME, True, True, True, False, 60)


## 9 · Mapas de calor
Densidad de ocupación por equipo, suavizada con gaussiana.

In [22]:
# ═══════════════════════════════════════════════════════════
#  MAPAS DE CALOR POR EQUIPO / ROL
# ═══════════════════════════════════════════════════════════

import cv2 as _cv2_hm
from scipy.ndimage import gaussian_filter

# ── Parametros de suavizado ───────────────────────────────────────────────
SIGMA_PX = 8    # suavizado gaussiano (en px del mapa, 1 px = 1/ESCALA metros)

# ── Agrupar heatmaps por equipo ───────────────────────────────────────────
def _combine(*keys):
    out = np.zeros((HM_H, HM_W), dtype=np.float32)
    for k in keys:
        if k in heatmaps:
            out += heatmaps[k]
    return gaussian_filter(out, sigma=SIGMA_PX)

hm_home  = _combine('player_home', 'gk_home')
hm_away  = _combine('player_away', 'gk_away')
hm_ref   = _combine('referee')

groups = [
    ('Local',     hm_home,  'Reds',    'player_home'),
    ('Visitante', hm_away,  'Blues',   'player_away'),
    ('Arbitro',   hm_ref,   'Greens',  'referee'),
]
# Filtrar grupos vacios
groups = [(lbl, hm, cmap, cls)
          for lbl, hm, cmap, cls in groups if hm.max() > 0]

if not groups:
    print("No hay datos suficientes para mapas de calor.")
else:
    ncols = len(groups)
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    fig.patch.set_facecolor(DARK_BG)
    if ncols == 1:
        axes = [axes]

    for ax, (lbl, hm, cmap, cls) in zip(axes, groups):
        draw_pitch_mpl(ax, L_M, A_M)

        # Normalizar y superponer heatmap
        hm_norm = hm / hm.max() if hm.max() > 0 else hm
        extent = [0, L_M, A_M, 0]   # xmin, xmax, ymax, ymin (y=0 arriba)
        ax.imshow(hm_norm, extent=extent, origin='upper',
                  cmap=cmap, alpha=0.65, vmin=0, vmax=1, zorder=4)

        # Punto caliente
        idx = np.unravel_index(np.argmax(hm), hm.shape)
        hotx = (idx[1] + 0.5) / ESCALA
        hoty = (idx[0] + 0.5) / ESCALA
        ax.scatter(hotx, hoty, marker='*', s=200,
                   color=COLOR_MAP_HEX.get(cls, 'white'),
                   edgecolors='white', lw=0.8, zorder=6,
                   label=f'Max ({hotx:.0f},{hoty:.0f}) m')

        ax.set_title(f'Equipo {lbl}', color='white', fontsize=11,
                     fontweight='bold')
        ax.legend(fontsize=8, facecolor='#1a1a2e', labelcolor='white',
                  edgecolor='#333', framealpha=0.8)

    total_s = N_FRAMES / FPS
    fig.suptitle(
        f"Mapas de calor  —  {total_s:.0f} s procesados  "
        f"({START_FRAME}–{END_FRAME})",
        color='white', fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()


C:\Users\xavia\AppData\Local\Temp\ipykernel_28828\361814029.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10 · Estadísticas
Distancia recorrida por jugador, distribución por clase y ocupación de zonas.

In [23]:
# ═══════════════════════════════════════════════════════════
#  ESTADISTICAS DE JUGADORES
#  Distancia recorrida · Ocupacion por zona · Distribucion
# ═══════════════════════════════════════════════════════════

if len(df) == 0:
    print("Sin datos para estadísticas. Ejecuta primero la celda de procesado.")
else:
    # ── 1. Distancia total por jugador ────────────────────────────────────
    def _total_distance(grp):
        grp = grp.sort_values('frame')
        dx  = grp['x_m'].diff().fillna(0)
        dy  = grp['y_m'].diff().fillna(0)
        # Filtrar saltos grandes (teleportaciones entre camaras/frames perdidos)
        dist = np.sqrt(dx**2 + dy**2)
        dist[dist > TRACK_MAX_DIST_M * 2] = 0
        return dist.sum()

    dist_df = (df.groupby(['gid', 'cls'])
                 .apply(_total_distance)
                 .reset_index(name='dist_m'))
    dist_df = dist_df.sort_values('dist_m', ascending=False)

    # ── 2. Ocupacion de zonas (tercio del campo) ──────────────────────────
    def _zone(x_m):
        if x_m < L_M / 3:          return 'Zona defensiva'
        elif x_m < 2 * L_M / 3:    return 'Zona media'
        else:                       return 'Zona ofensiva'

    df['zone'] = df['x_m'].apply(_zone)
    zone_df = (df.groupby(['cls', 'zone'])
                 .size()
                 .reset_index(name='count'))

    # ── 3. Figura con 3 paneles ───────────────────────────────────────────
    fig = plt.figure(figsize=(18, 10))
    fig.patch.set_facecolor(DARK_BG)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ax_dist  = fig.add_subplot(gs[0, :2])   # Distancia top-N jugadores
    ax_kde   = fig.add_subplot(gs[0, 2])    # Distribucion de distancias
    ax_zone  = fig.add_subplot(gs[1, :])    # Ocupacion por zona

    # Panel 1: distancia top-20
    top_n = dist_df.head(20)
    colors_bar = [COLOR_MAP_HEX.get(c, '#aaa') for c in top_n['cls']]
    bars = ax_dist.barh(
        [f"#{int(r.gid)} ({r.cls[:3]})" for r in top_n.itertuples()],
        top_n['dist_m'],
        color=colors_bar, edgecolor='#333', linewidth=0.5
    )
    ax_dist.set_xlabel('Distancia recorrida (m)', color='#aaa')
    ax_dist.set_title('Top 20 — Distancia por jugador', color='white',
                       fontsize=10, fontweight='bold')
    ax_dist.set_facecolor(DARK_BG)
    ax_dist.tick_params(colors='#aaa', labelsize=8)
    ax_dist.invert_yaxis()
    ax_dist.grid(axis='x', alpha=0.15, color='#555')
    for sp in ax_dist.spines.values(): sp.set_color('#333')

    # Panel 2: distribucion de distancias por clase (violinplot approx)
    cls_list  = [c for c in CLASSES_ORDER if c in dist_df['cls'].values]
    cls_data  = [dist_df[dist_df['cls'] == c]['dist_m'].values for c in cls_list]
    cls_data  = [d for d in cls_data if len(d) > 0]
    cls_names = [c for c, d in zip(cls_list, cls_data) if len(d) > 0]

    if cls_data:
        vp = ax_kde.violinplot(cls_data, showmedians=True, showextrema=True)
        for i, (body, cls_n) in enumerate(zip(vp['bodies'], cls_names)):
            body.set_facecolor(COLOR_MAP_HEX.get(cls_n, '#aaa'))
            body.set_alpha(0.7)
        ax_kde.set_xticks(range(1, len(cls_names) + 1))
        ax_kde.set_xticklabels([c[:4] for c in cls_names],
                                fontsize=8, color='#aaa')
    ax_kde.set_ylabel('Distancia (m)', color='#aaa')
    ax_kde.set_title('Distribucion por clase', color='white',
                      fontsize=10, fontweight='bold')
    ax_kde.set_facecolor(DARK_BG)
    ax_kde.tick_params(colors='#aaa', labelsize=8)
    ax_kde.grid(axis='y', alpha=0.15, color='#555')
    for sp in ax_kde.spines.values(): sp.set_color('#333')

    # Panel 3: ocupacion de zonas (barras apiladas)
    zones  = ['Zona defensiva', 'Zona media', 'Zona ofensiva']
    z_colors = ['#ef4444', '#f59e0b', '#22c55e']
    bottom = np.zeros(len(CLASSES_ORDER))
    cls_idx = {c: i for i, c in enumerate(CLASSES_ORDER)}

    for zone, zcolor in zip(zones, z_colors):
        vals = []
        for cls_n in CLASSES_ORDER:
            sub = zone_df[(zone_df['cls'] == cls_n) & (zone_df['zone'] == zone)]
            vals.append(sub['count'].sum() if len(sub) else 0)
        vals = np.array(vals, dtype=float)
        # Normalizar a % dentro de cada clase
        totals = zone_df.groupby('cls')['count'].sum()
        for i, cls_n in enumerate(CLASSES_ORDER):
            if cls_n in totals.index and totals[cls_n] > 0:
                vals[i] = 100.0 * vals[i] / totals[cls_n]
        ax_zone.bar(CLASSES_ORDER, vals, bottom=bottom,
                    color=zcolor, label=zone, alpha=0.85)
        bottom += vals

    ax_zone.set_ylim(0, 100)
    ax_zone.set_ylabel('% de detecciones', color='#aaa')
    ax_zone.set_title('Ocupacion por zona del campo (% de tiempo)', color='white',
                       fontsize=10, fontweight='bold')
    ax_zone.set_facecolor(DARK_BG)
    ax_zone.tick_params(colors='#aaa', labelsize=9)
    ax_zone.legend(fontsize=8, facecolor='#1a1a2e', labelcolor='white',
                   edgecolor='#333', framealpha=0.8)
    ax_zone.grid(axis='y', alpha=0.15, color='#555')
    for sp in ax_zone.spines.values(): sp.set_color('#333')
    # Colorear ticks por clase
    for tick, cls_n in zip(ax_zone.get_xticklabels(), CLASSES_ORDER):
        tick.set_color(COLOR_MAP_HEX.get(cls_n, 'white'))

    total_s = N_FRAMES / FPS
    fig.suptitle(
        f"Estadisticas de tracking  —  {total_s:.0f} s  "
        f"({df['gid'].nunique()} IDs unicos)",
        color='white', fontsize=13, fontweight='bold'
    )
    plt.show()

    # ── Resumen textual ────────────────────────────────────────────────────
    print("\n── Distancia media por clase (m) ──")
    print(dist_df.groupby('cls')['dist_m'].mean().round(1).to_string())
    print(f"\n── Total IDs detectados: {df['gid'].nunique()} ──")
    print(df.groupby('cls')['gid'].nunique().to_string())



── Distancia media por clase (m) ──
cls
gk_away        27.100000
gk_home        10.000000
player_away    33.099998
player_home    24.600000
referee        15.300000
unknown         0.000000

── Total IDs detectados: 50 ──
cls
gk_away         2
gk_home         9
player_away    26
player_home    26
referee         5
unknown         2


C:\Users\xavia\AppData\Local\Temp\ipykernel_28828\2753869773.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_total_distance)
C:\Users\xavia\AppData\Local\Temp\ipykernel_28828\2753869773.py:124: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11 · Exportación
Guarda trayectorias en CSV. Activa `EXPORT_VIDEO = True` para generar el video anotado (tarda varios minutos).

In [24]:
# ═══════════════════════════════════════════════════════════
#  EXPORTACION
#  CSV de trayectorias · Video anotado (opcional)
# ═══════════════════════════════════════════════════════════

import os as _os

# ── 1. CSV de trayectorias ────────────────────────────────────────────────
CSV_PATH = f"tracking_{Path(VIDEO_PATH).stem}_f{START_FRAME}-{END_FRAME}.csv"

if len(df):
    df.to_csv(CSV_PATH, index=False)
    print(f"CSV guardado: {CSV_PATH}  ({len(df)} filas, {df['gid'].nunique()} IDs)")
else:
    print("Sin datos — CSV no generado.")

# ── 2. Video anotado (opcional — puede tardar varios minutos) ─────────────
EXPORT_VIDEO = False   # Cambia a True para exportar

if EXPORT_VIDEO and len(df):
    OUT_VIDEO = f"tracking_{Path(VIDEO_PATH).stem}_f{START_FRAME}-{END_FRAME}.mp4"
    _fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    _out    = cv2.VideoWriter(OUT_VIDEO, _fourcc, FPS, (VID_W, VID_H))

    cap_exp = cv2.VideoCapture(VIDEO_PATH)
    cap_exp.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

    print(f"Exportando video anotado → {OUT_VIDEO}  ({N_FRAMES} frames)…")
    for _fi in range(N_FRAMES):
        _ret, _frame = cap_exp.read()
        if not _ret:
            break
        _fidx = START_FRAME + _fi
        _protos_e = select_protos(_frame, protos_day, protos_night)

        # Anotar Cam A
        _cam_a = preprocess_half(_frame[:HALF_H, :], 'left')
        _res_a = model_A.predict(_cam_a, conf=CONF_THRESH, verbose=False)
        if _res_a and _res_a[0].boxes is not None:
            for _box in _res_a[0].boxes:
                _x1, _y1, _x2, _y2 = [int(v) for v in _box.xyxy[0]]
                _crop = _cam_a[max(0,_y1):_y2, max(0,_x1):_x2]
                if _crop.size == 0: continue
                _cn, _ = classify_player(get_torso_crop(_crop), _protos_e)
                _bgr   = COLOR_MAP_BGR.get(_cn, (200, 200, 200))
                cv2.rectangle(_frame, (_x1, _y1), (_x2, _y2), _bgr, 2)
                # Buscar gid
                _df_row = df[(df['frame'] == _fidx) & (df['cam'] == 'A')]
                if len(_df_row):
                    _cx = (_x1+_x2)/2; _cy = float(_y2)
                    try: _xm, _ym = pixel_to_field(_cx, _cy, H_a)
                    except: _xm, _ym = None, None
                    if _xm:
                        _dists = np.sqrt((_df_row['x_m']-_xm)**2 + (_df_row['y_m']-_ym)**2)
                        if _dists.min() < 3.0:
                            _gid = int(_df_row.iloc[_dists.argmin()]['gid'])
                            cv2.putText(_frame, f"#{_gid}", (_x1, _y1-5),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, _bgr, 2)

        # Anotar Cam B (mismo patron, offset Y)
        _cam_b = preprocess_half(_frame[HALF_H:,  :], 'right')
        _res_b = model_B.predict(_cam_b, conf=CONF_THRESH, verbose=False)
        if _res_b and _res_b[0].boxes is not None:
            for _box in _res_b[0].boxes:
                _x1, _y1, _x2, _y2 = [int(v) for v in _box.xyxy[0]]
                _y1o, _y2o = _y1 + HALF_H, _y2 + HALF_H   # offset en frame completo
                _crop = _cam_b[max(0,_y1):_y2, max(0,_x1):_x2]
                if _crop.size == 0: continue
                _cn, _ = classify_player(get_torso_crop(_crop), _protos_e)
                _bgr   = COLOR_MAP_BGR.get(_cn, (200, 200, 200))
                cv2.rectangle(_frame, (_x1, _y1o), (_x2, _y2o), _bgr, 2)

        _out.write(_frame)
        if (_fi + 1) % 150 == 0:
            print(f"  {100*(_fi+1)/N_FRAMES:.0f}%  frame {_fidx}")

    cap_exp.release()
    _out.release()
    print(f"Video exportado: {OUT_VIDEO}  ({_os.path.getsize(OUT_VIDEO)//1024} KB)")

elif EXPORT_VIDEO:
    print("Sin datos de tracking — exportacion cancelada.")
else:
    print("Exportacion de video desactivada (EXPORT_VIDEO=False).")


CSV guardado: tracking_veo_panoramico_banyoles_f18000-18900.csv  (14186 filas, 50 IDs)
Exportacion de video desactivada (EXPORT_VIDEO=False).
